### Loading Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/online_retail_II.xlsx"

sheets = pd.read_excel(DATA_PATH, sheet_name=None, engine="openpyxl")
list(sheets.keys())

['Year 2009-2010', 'Year 2010-2011']

### Do the sheets share a schema and a date range?

In [2]:
for name, sheet_df in sheets.items():
    print(name, sheet_df.shape, sheet_df["InvoiceDate"].min(), sheet_df["InvoiceDate"].max())

sheets["Year 2009-2010"].columns.equals(sheets["Year 2010-2011"].columns)

Year 2009-2010 (525461, 8) 2009-12-01 07:45:00 2010-12-09 20:01:00
Year 2010-2011 (541910, 8) 2010-12-01 08:26:00 2011-12-09 12:50:00


True

### How many invoice numbers appear in both sheets?

In [3]:
overlap = set(sheets["Year 2009-2010"]["Invoice"]) & set(sheets["Year 2010-2011"]["Invoice"])
print(f"{len(overlap)} invoice numbers appear in both sheets")

1088 invoice numbers appear in both sheets


### Are shared invoices identical rows or different transactions?

In [4]:
print(sheets["Year 2009-2010"][sheets["Year 2009-2010"]["Invoice"] == 536365])
print(sheets["Year 2010-2011"][sheets["Year 2010-2011"]["Invoice"] == 536365])

       Invoice StockCode                          Description  Quantity  \
502938  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
502939  536365     71053                  WHITE METAL LANTERN         6   
502940  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
502941  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
502942  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
502943  536365     22752         SET 7 BABUSHKA NESTING BOXES         2   
502944  536365     21730    GLASS STAR FROSTED T-LIGHT HOLDER         6   

               InvoiceDate  Price  Customer ID         Country  
502938 2010-12-01 08:26:00   2.55      17850.0  United Kingdom  
502939 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502940 2010-12-01 08:26:00   2.75      17850.0  United Kingdom  
502941 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502942 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502943 20

### How many duplicate rows would a naive concat produce?

In [5]:
combined = pd.concat(sheets.values(), ignore_index=True)
combined.duplicated().sum()

np.int64(34335)

### Where in time do the shared invoices sit?

In [6]:
overlap_in_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"]["Invoice"].isin(overlap)]["InvoiceDate"]
overlap_in_2010 = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["Invoice"].isin(overlap)]["InvoiceDate"]

print("In 2009-2010 sheet:", overlap_in_2009.min(), "to", overlap_in_2009.max())
print("In 2010-2011 sheet:", overlap_in_2010.min(), "to", overlap_in_2010.max())

In 2009-2010 sheet: 2010-12-01 08:26:00 to 2010-12-09 20:01:00
In 2010-2011 sheet: 2010-12-01 08:26:00 to 2010-12-09 20:01:00



### How many rows fall inside that window in each sheet?

In [7]:
window_start = overlap_in_2010.min()
window_end = overlap_in_2010.max()

rows_in_window_2009 = sheets["Year 2009-2010"]["InvoiceDate"].between(window_start, window_end).sum()
rows_in_window_2010 = sheets["Year 2010-2011"]["InvoiceDate"].between(window_start, window_end).sum()

print("Rows in window, 2009-2010 sheet:", rows_in_window_2009)
print("Rows in window, 2010-2011 sheet:", rows_in_window_2010)

Rows in window, 2009-2010 sheet: 22523
Rows in window, 2010-2011 sheet: 22523


### Is the second sheet's window a pure copy, and what duplicates exist inside each sheet?

In [8]:
window_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"]["InvoiceDate"].between(window_start, window_end)]
window_2010 = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["InvoiceDate"].between(window_start, window_end)]

print("Unique rows, 2009-2010 window:       ", len(window_2009.drop_duplicates()))
print("Unique rows, both windows combined:  ", len(pd.concat([window_2009, window_2010]).drop_duplicates()))
print("Exact duplicates inside 2009-2010 sheet:", sheets["Year 2009-2010"].duplicated().sum())
print("Exact duplicates inside 2010-2011 sheet:", sheets["Year 2010-2011"].duplicated().sum())

Unique rows, 2009-2010 window:        22202
Unique rows, both windows combined:   22202
Exact duplicates inside 2009-2010 sheet: 6865
Exact duplicates inside 2010-2011 sheet: 5268


### What do the within-sheet duplicates look like?

In [9]:
cols = ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]

dups_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"].duplicated(keep=False)]

print("Invoices affected:", dups_2009["Invoice"].nunique())
print(dups_2009[cols].head(10))

Invoices affected: 2538
    Invoice StockCode                        Description  Quantity  Price  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75   
363  489517     21912           VINTAGE SNAKES & LADDERS         1   3.75   
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   3.75   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   0.65   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   0.85   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   3.75   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   1.95   
383  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   0.85   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   0.65   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75   

     Customer ID  
362      16329.0  
363      16329.0  
365      16329.0  
367      16329.0  
368      16329.0  
371      16329

### How much value do the within-sheet duplicates carry?

In [10]:
for name, sheet_df in sheets.items():
    extra_copy = sheet_df.duplicated()
    line_total = sheet_df["Quantity"] * sheet_df["Price"]
    share = line_total[extra_copy].sum() / line_total.sum() * 100
    print(f"{name} | extra-copy rows: {extra_copy.sum()} | share of net value: {share:.2f}%")

Year 2009-2010 | extra-copy rows: 6865 | share of net value: 0.35%
Year 2010-2011 | extra-copy rows: 5268 | share of net value: 0.22%


### Observations

**Findings:** The two sheets aren't a clean split. They have the same columns, but the
date ranges overlap from 2010-12-01 08:26 to 2010-12-09 20:01. That window has 22,523
rows in each sheet and 22,202 unique, and putting the two windows together adds nothing
new, so the 2010-2011 sheet just repeats the end of 2009-2010. 1,088 invoice numbers
show up in both, and invoice 536365 has identical lines, prices and customer in each. A
plain concat gives 34,335 duplicate rows: 22,202 from the overlap, 6,865 inside
2009-2010 and 5,268 inside 2010-2011.

The 12,133 duplicates inside single sheets are a separate thing. In the first two
invoices I looked at, the repeated lines were spread through the invoice, not sitting
next to each other, with normal quantities (1, 6, 12). They're worth 0.35% and 0.22% of
net value.

**Decision:** Keep all of 2009-2010 and only take 2010-2011 rows after 2010-12-09 20:01.
That drops the overlap (22,523 rows) and nothing gets counted twice. I'm leaving the
within-sheet repeats in.

**Why:** The overlap is definitely an artefact, so dropping it was easy. The
within-sheet repeats are harder. There's no line number column, so I can't tell a
logging error from someone genuinely adding the same item twice. The sample looked more
like items added at different points in an order than a repeated log line, but that's
only two invoices, so I'm not claiming more than that. They're also tiny, under 0.4% of
value, so deleting them on a guess could remove real purchases for almost no gain.

**What I'm accepting:** If they are logging errors, Monetary is overstated by roughly
0.2 to 0.35% overall, and more for any customer with lots of repeated lines. Frequency
should be fine if it counts distinct invoices. I can check later by re-running the
clustering without the repeats and seeing if the segments move.

### Building df: both sheets, minus the overlap

In [11]:
# Last timestamp in 2009-2010. The 2010-2011 sheet repeats everything up to here,
# so only rows strictly after it are new.
OVERLAP_CUTOFF = sheets["Year 2009-2010"]["InvoiceDate"].max()

early = sheets["Year 2009-2010"]
late_new_rows = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["InvoiceDate"] > OVERLAP_CUTOFF]

df = pd.concat([early, late_new_rows], ignore_index=True)

print(df.shape)
print(df.duplicated().sum())

(1044848, 8)
11812


### Is Invoice a consistent type?

In [12]:
print(df["Invoice"].map(type).value_counts())

df["Invoice"] = df["Invoice"].astype(str)

print(df["Invoice"].map(type).value_counts())

Invoice
<class 'int'>    1025677
<class 'str'>      19171
Name: count, dtype: int64
Invoice
<class 'str'>    1044848
Name: count, dtype: int64


### What did df actually load as?

In [13]:
print(df.shape)
print(df.dtypes)
df.head()

(1044848, 8)
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Summary statistics

In [14]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,1044848,53628,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,1044848,5305,85123A,5711,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,1040573,5698,WHITE HANGING HEART T-LIGHT HOLDER,5798,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,1044848.0,NaN,NaN,NaN,9.993649,-80995.0,1.0,3.0,10.0,80995.0,174.218493
InvoiceDate,1044848,NaN,NaN,NaN,2011-01-03 11:44:15.916841728,2009-12-01 07:45:00,2010-07-05 11:11:00,2010-12-08 16:34:00,2011-07-27 13:42:00,2011-12-09 12:50:00,NaN
Price,1044848.0,NaN,NaN,NaN,4.590546,-53594.36,1.25,2.1,4.13,38970.0,121.704167
Customer ID,809561.0,NaN,NaN,NaN,15319.760422,12346.0,13969.0,15248.0,16792.0,18287.0,1695.812057
Country,1044848,43,United Kingdom,959983,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### How much is missing in each column?

In [15]:
print(df.isna().sum())
print((df.isna().mean() * 100).round(2))

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235287
Country             0
dtype: int64
Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.52
Country         0.00
dtype: float64


### How many rows hit each flag?

In [16]:
is_cancelled = df["Invoice"].str.startswith("C")
is_negative_qty = df["Quantity"] < 0
is_null_customer = df["Customer ID"].isna()
is_nonpositive_price = df["Price"] <= 0

print("Cancelled invoices:", is_cancelled.sum())
print("Negative Quantity: ", is_negative_qty.sum())
print("Null Customer ID:  ", is_null_customer.sum())
print("Price <= 0:        ", is_nonpositive_price.sum())

Cancelled invoices: 19165
Negative Quantity:  22557
Null Customer ID:   235287
Price <= 0:         6029


### Do cancelled orders and negative quantities overlap?

In [17]:
print(pd.crosstab(is_cancelled, is_negative_qty))

Quantity    False  True 
Invoice                 
False     1022290   3393
True            1  19164


### What are the zero-price rows?

In [18]:
df.loc[df["Price"] == 0, ["Invoice", "StockCode", "Description", "Quantity", "Price"]].drop_duplicates("StockCode").head(20)

,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,NaN,-50,0.0
3114,489655,20683,NaN,-44,0.0
3161,489659,21350,NaN,230,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
3731,489781,84292,NaN,17,0.0
4296,489806,18010,NaN,-770,0.0


### What are the negative-price rows?

In [19]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


### Which invoices are neither plain numbers nor cancellations?

In [20]:
df[~is_cancelled & ~df["Invoice"].str.isdigit()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802920,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


### What is the one cancelled row with a non-negative quantity?

In [21]:
df[is_cancelled & ~is_negative_qty]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


### Are the non-cancelled negative quantities zero-price rows?

In [22]:
print(pd.crosstab(is_negative_qty & ~is_cancelled, is_nonpositive_price))

Price    False  True 
row_0                
False  1038819   2636
True         0   3393


### What share of value has no Customer ID?

In [23]:
line_total = df["Quantity"] * df["Price"]
print(f"{line_total[is_null_customer].sum() / line_total.sum() * 100:.2f}% of net value has no Customer ID")

13.57% of net value has no Customer ID


### Which StockCodes are not ordinary product codes?

In [24]:
df["StockCode"] = df["StockCode"].astype(str)

non_product = df[~df["StockCode"].str.match(r"^\d{5}")]
non_product["StockCode"].value_counts().head(15)

StockCode
POST            2086
DOT             1425
M               1398
C2               277
D                173
S                102
BANK CHARGES     100
ADJUST            67
AMAZONFEE         36
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64

### What are the descriptions behind these codes?

In [25]:
codes = non_product["StockCode"].value_counts().head(15).index
non_product.groupby("StockCode")["Description"].first().loc[codes]

StockCode
POST                                        POSTAGE
DOT                                  DOTCOM POSTAGE
M                                            Manual
C2                                         CARRIAGE
D                                          Discount
S                                           SAMPLES
BANK CHARGES                           Bank Charges
ADJUST          Adjustment by john on 26/01/2010 16
AMAZONFEE                                AMAZON FEE
DCGS0058                           MISO PRETTY  GUM
gift_0001_20     Dotcomgiftshop Gift Voucher £20.00
gift_0001_30     Dotcomgiftshop Gift Voucher £30.00
DCGSSGIRL                                    update
DCGSSBOY                                     update
PADS                     PADS TO MATCH ALL CUSHIONS
Name: Description, dtype: object

### How much identified-customer value sits in non-product rows?

In [26]:
is_non_product = ~df["StockCode"].str.match(r"^\d{5}")
identified = ~is_null_customer
line_total = df["Quantity"] * df["Price"]

print("Non-product rows with a Customer ID:", (is_non_product & identified).sum())
print(f"Share of identified customers' net value: {line_total[is_non_product & identified].sum() / line_total[identified].sum() * 100:.2f}%")

Non-product rows with a Customer ID: 3673
Share of identified customers' net value: -0.42%


### What is left of each flag once null Customer IDs are removed?

In [27]:
identified_rows = df[~is_null_customer]

print("Identified rows:", len(identified_rows))
print("Cancelled:      ", identified_rows["Invoice"].str.startswith("C").sum())
print("Price <= 0:     ", (identified_rows["Price"] <= 0).sum())
print("Non-product:    ", (~identified_rows["StockCode"].str.match(r"^\d{5}")).sum())

Identified rows: 809561
Cancelled:       18446
Price <= 0:      70
Non-product:     3673


### How big are cancellations for identified customers?

In [28]:
cancelled_identified = identified_rows["Invoice"].str.startswith("C")
identified_totals = identified_rows["Quantity"] * identified_rows["Price"]

cancelled_value = identified_totals[cancelled_identified].sum()
sales_value = identified_totals[~cancelled_identified].sum()

print(f"Cancelled value:     {cancelled_value:,.0f}")
print(f"Non-cancelled value: {sales_value:,.0f}")
print(f"Cancelled as a share of non-cancelled value: {abs(cancelled_value) / sales_value * 100:.2f}%")
print("Customers with at least one cancellation:",
      identified_rows.loc[cancelled_identified, "Customer ID"].nunique(),
      "of", identified_rows["Customer ID"].nunique())

Cancelled value:     -1,087,882
Non-cancelled value: 17,431,551
Cancelled as a share of non-cancelled value: 6.24%
Customers with at least one cancellation: 2572 of 5942


### What do cancellations do at the customer level?

In [29]:
net_by_customer = identified_totals.groupby(identified_rows["Customer ID"]).sum()
purchasers = set(identified_rows.loc[~cancelled_identified, "Customer ID"])
all_customers = set(identified_rows["Customer ID"])

print("Customers with net spend <= 0:      ", (net_by_customer <= 0).sum())
print("Customers with only cancellations:  ", len(all_customers - purchasers))

Customers with net spend <= 0:       99
Customers with only cancellations:   61


### What are the Price <= 0 rows left for identified customers

In [30]:
identified_rows[identified_rows["Price"] <= 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
...,...,...,...,...,...,...,...,...
982017,577129,22464,HANGING METAL HEART LANTERN,4,2011-11-17 19:52:00,0.0,15602.0,United Kingdom
982491,577168,M,Manual,1,2011-11-18 10:42:00,0.0,12603.0,Germany
983587,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,0.0,12444.0,Norway
988923,577696,M,Manual,1,2011-11-21 11:57:00,0.0,16406.0,United Kingdom


### Observations

**Findings:** Null Customer IDs account for 235,287 rows (22.5% of rows, 13.57% of
net value). Once they're removed, 809,561 rows remain. Of those, 18,446 are
cancellations (about -£1.09M against £17.4M of non-cancelled value, 6.24%), and they
touch 2,572 of 5,942 customers. 99 customers have net spend of zero or less, and 61 of
them only have cancellations. Only 70 rows have Price <= 0, because most zero-price and
stock-adjustment rows were guest orders. Those 70 are a mix of Manual lines and real
products at £0, so probably freebies or pricing gaps. 3,673 rows are non-product codes
(postage, fees, discounts, manual lines), worth -0.42% of identified value. The cleaned
frame has 809,491 rows and 5,939 customers.

**What Monetary means here:** Net spend. I want it to show what a customer is worth to
the business, so returns count against them.

**Decisions**

1. **Null Customer ID:** dropped. There's no customer to attach them to, so the
   segmentation only covers identified customers (77.5% of rows).
2. **Cancelled orders:** kept as negative lines in Monetary, but not counted toward
   Frequency or Recency, since a return isn't a purchase or a visit. The 61 customers
   with only cancellations have no purchases to segment, so they come out when I build
   RFM.
3. **Price <= 0:** dropped. They add £0 to Monetary anyway, and 70 rows is under 0.01%
   of the data. This removes 3 customers who only had £0 rows.
4. **Non-product codes:** kept. Postage and fees are money the customer paid, the effect
   is -0.42%, and my "not 5 digits" rule also catches real items (PADS, DCGS0058), so
   dropping on it could remove genuine rows.

**What I'm accepting:** 38 customers who did buy something still end up with net spend
of zero or less. That needs handling before any log transform in Step 3, and K-Means may
treat them as outliers. Net spend also doesn't fix how skewed spend is, and keeping
postage and discounts means Monetary isn't purely product demand. Dropping the £0
product rows can slightly undercount activity for a customer whose only invoice on a day
was a freebie.

In [31]:
# Cancellations and non-product codes are kept on purpose: the RFM build in Step 2 uses
# cancellations for Monetary only, and the non-product rows are -0.42% of value.
df_clean = df[df["Customer ID"].notna() & (df["Price"] > 0)].copy()

# Customer ID was a float only because of the nulls
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)

print(df_clean.shape)
print(df_clean["Customer ID"].nunique())

(809491, 8)
5939
